# Import the Libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, Flatten, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

# Load and preprocess the CIFAR-10 dataset


In [2]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

170498071/170498071 [==============================] - 2s 0us/step


# Resize the images to match ResNet50 input dimensions (224x224)

In [3]:
x_train_resized = tf.image.resize(x_train, (224, 224))
x_test_resized = tf.image.resize(x_test, (224, 224))

# Normalize the pixel values to [0, 1]

In [4]:
x_train_resized = x_train_resized / 255.0
x_test_resized = x_test_resized / 255.0

# One-hot encode the labels

In [5]:
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Load the ResNet50 model, excluding the top fully connected layers

In [6]:
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

94765736/94765736 [==============================] - 0s 0us/step


# Freeze the base model layers to retain pre-trained weights

In [7]:
for layer in base_model.layers:
    layer.trainable = False

# Add custom classification layers

In [8]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dense(128, activation='relu')(x)
output = Dense(10, activation='softmax')(x)  # CIFAR-10 has 10 classes

# Create the final model

In [9]:
model = Model(inputs=base_model.input, outputs=output)

# Compile the model


In [10]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model

In [11]:
model.fit(x_train_resized, y_train, epochs=10, validation_data=(x_test_resized, y_test))

Epoch 1/10
1563/1563 [==============================] - 1008s 643ms/step - loss: 2.3059 - accuracy: 0.1043 - val_loss: 2.3087 - val_accuracy: 0.1020
Epoch 2/10
1563/1563 [==============================] - 1016s 650ms/step - loss: 2.2442 - accuracy: 0.1414 - val_loss: 2.2176 - val_accuracy: 0.1540
Epoch 3/10
1563/1563 [==============================] - 1018s 651ms/step - loss: 2.1951 - accuracy: 0.1704 - val_loss: 2.1389 - val_accuracy: 0.1755
Epoch 4/10
1563/1563 [==============================] - 1023s 655ms/step - loss: 2.0492 - accuracy: 0.2285 - val_loss: 1.9743 - val_accuracy: 0.2606
Epoch 5/10
1563/1563 [==============================] - 994s 636ms/step - loss: 1.9712 - accuracy: 0.2524 - val_loss: 1.9336 - val_accuracy: 0.2661
Epoch 6/10
1563/1563 [==============================] - 1023s 655ms/step - loss: 1.9439 - accuracy: 0.2594 - val_loss: 1.9105 - val_accuracy: 0.2712
Epoch 7/10
1563/1563 [==============================] - 1042s 667ms/step - loss: 1.9213 - accuracy: 0.2667 

# Evaluate the model


In [12]:
loss, accuracy = model.evaluate(x_test_resized, y_test)
print(f"ResNet50 - Loss: {loss}, Accuracy: {accuracy}")

313/313 [==============================] - 164s 523ms/step - loss: 1.8976 - accuracy: 0.2758
ResNet50 - Loss: 1.8976032733917236, Accuracy: 0.2757999897003174
